# Labelling Evals Runner

Runs the labelling orchestrator across iterations defined in `mappings/evals.json`.  
Each iteration specifies a model, reasoning effort, and per-task train/dev/test configs.

In [1]:
ITERATION_ID = 1

# Which tasks to run (None = all tasks in the iteration)
# e.g. ["screening", "driver"] or None for all
# TASKS_TO_RUN = ["screening", "driver"] 
TASKS_TO_RUN = None

SPLITS_TO_RUN = ["train", "dev", "test"]
SUBMISSION_MODE = "live"
LIMIT_DOCS = None
EVALS_JSON_PATH = "mappings/evals.json"

In [2]:
import json
import os
import subprocess
from pathlib import Path

# Load evals config
with open(EVALS_JSON_PATH, "r") as f:
    evals_config = json.load(f)

# Find the selected iteration
iteration = None
for it in evals_config["iterations"]:
    if it["iteration_id"] == ITERATION_ID:
        iteration = it
        break

if iteration is None:
    available = [it["iteration_id"] for it in evals_config["iterations"]]
    raise ValueError(f"Iteration {ITERATION_ID} not found. Available: {available}")

model = iteration["model"]
reasoning = iteration["reasoning"]["effort"]
tasks = iteration["tasks"]

# Filter tasks if specified
if TASKS_TO_RUN is not None:
    tasks = {k: v for k, v in tasks.items() if k in TASKS_TO_RUN}

print(f"Iteration:  {ITERATION_ID} — {iteration['description']}")
print(f"Model:      {model}")
print(f"Reasoning:  {reasoning}")
print(f"Tasks:      {list(tasks.keys())}")
print(f"Splits:     {SPLITS_TO_RUN}")
print()

# Build the list of (task_name, split, data_path, run_name) runs
runs = []
for task_name, splits in tasks.items():
    for split in SPLITS_TO_RUN:
        if split not in splits:
            print(f"  WARNING: split '{split}' not found for task '{task_name}', skipping")
            continue
        cfg = splits[split]
        runs.append((task_name, split, cfg["data_path"], cfg["run_name"]))

print(f"Total runs: {len(runs)}")
for task_name, split, data_path, run_name in runs:
    print(f"  {task_name:12s} {split:5s}  {data_path}  →  {run_name}")

Iteration:  1 — gpt-5-nano low reasoning
Model:      gpt-5-nano-2025-08-07
Reasoning:  low
Tasks:      ['screening', 'driver', 'threat', 'geography', 'ecosystem', 'study']
Splits:     ['train', 'dev', 'test']

Total runs: 18
  screening    train  data/labels/l0/train  →  l0-train-gpt5nanolow-25022026-01
  screening    dev    data/labels/l0/dev  →  l0-dev-gpt5nanolow-25022026-01
  screening    test   data/labels/l0/test  →  l0-test-gpt5nanolow-25022026-01
  driver       train  data/labels/l1/train  →  l1-train-gpt5nanolow-25022026-01
  driver       dev    data/labels/l1/dev  →  l1-dev-gpt5nanolow-25022026-01
  driver       test   data/labels/l1/test  →  l1-test-gpt5nanolow-25022026-01
  threat       train  data/labels/l2/train  →  l2-train-gpt5nanolow-25022026-01
  threat       dev    data/labels/l2/dev  →  l2-dev-gpt5nanolow-25022026-01
  threat       test   data/labels/l2/test  →  l2-test-gpt5nanolow-25022026-01
  geography    train  data/labels/l3/train  →  l3-train-gpt5nanolow-25022

In [3]:
import re
import time
import sys

# Add labelling package to path so we can use poll/download utilities
sys.path.insert(0, "labelling")
from src.openai_batches import build_client, poll_batch, download_batch_output

BATCH_OUTPUTS_DIR = Path("labelling/artifacts/batch_outputs")

def run_orchestrator(task_name, data_path, run_name, model, reasoning):
    """Call the orchestrator as a subprocess with model/reasoning overrides.
    Returns (exit_code, list_of_batch_ids)."""
    env = os.environ.copy()
    env["ORCHESTRATOR_MODEL"] = model
    env["ORCHESTRATOR_REASONING"] = reasoning
    env["ORCHESTRATOR_SUBMISSION_MODE"] = SUBMISSION_MODE
    env["ORCHESTRATOR_LIMIT_DOCS"] = str(LIMIT_DOCS) if LIMIT_DOCS is not None else "none"

    cmd = [
        "python", "labelling/orchestrator.py",
        data_path,
        run_name,
        "--task", task_name,
    ]

    print(f"  CMD: {' '.join(cmd)}")
    print(f"  ENV overrides: MODEL={model}, REASONING={reasoning}, "
          f"MODE={SUBMISSION_MODE}, LIMIT_DOCS={LIMIT_DOCS}")
    print()

    result = subprocess.run(cmd, env=env, capture_output=True, text=True)

    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

    # Extract batch IDs from stdout (format: "Batch submitted: id=batch_xxx, ...")
    batch_ids = re.findall(r"Batch submitted: id=(batch_[^,\s]+)", result.stdout or "")

    if result.returncode != 0:
        print(f"  FAILED (exit code {result.returncode})")
    else:
        print(f"  DONE")
        if batch_ids:
            print(f"  Batch IDs: {batch_ids}")

    return result.returncode, batch_ids

In [ ]:
# ── Execute all runs ────────────────────────────────────────────────────────
results = []
all_batches = []  # collected batch submissions for polling later

for i, (task_name, split, data_path, run_name) in enumerate(runs, 1):
    print(f"{'='*60}")
    print(f"[{i}/{len(runs)}] task={task_name}  split={split}")
    print(f"{'='*60}")
    rc, batch_ids = run_orchestrator(task_name, data_path, run_name, model, reasoning)
    results.append({"task": task_name, "split": split, "run_name": run_name, "exit_code": rc})
    for bid in batch_ids:
        all_batches.append({
            "batch_id": bid,
            "task": task_name,
            "split": split,
            "run_name": run_name,
        })
    print()

# Summary
print(f"\n{'='*60}")
print("Summary")
print(f"{'='*60}")
for r in results:
    status = "OK" if r["exit_code"] == 0 else f"FAIL({r['exit_code']})"
    print(f"  {r['task']:12s} {r['split']:5s}  {status}  {r['run_name']}")

if all_batches:
    print(f"\nCollected {len(all_batches)} batch submission(s) for polling:")
    for b in all_batches:
        print(f"  {b['task']:12s} {b['split']:5s}  {b['batch_id']}  ({b['run_name']})")

[1/18] task=screening  split=train
  CMD: python labelling/orchestrator.py data/labels/l0/train l0-train-gpt5nanolow-25022026-01 --task screening
  ENV overrides: MODEL=gpt-5-nano-2025-08-07, REASONING=low, MODE=live, LIMIT_DOCS=None



## Poll & Download Batch Outputs

Run the cell below to poll all submitted batches until they complete, then download outputs to `labelling/artifacts/batch_outputs/{run_name}/`.

In [5]:
# ── Poll and download batch outputs ─────────────────────────────────────────
POLL_INTERVAL_SECONDS = 30  # how often to check batch status

if not all_batches:
    print("No batches to poll (either live mode was used or no runs produced batches).")
else:
    client = build_client()
    pending = list(all_batches)
    completed = []
    failed = []

    print(f"Polling {len(pending)} batch(es) every {POLL_INTERVAL_SECONDS}s ...\n")

    while pending:
        still_pending = []
        for entry in pending:
            status = poll_batch(entry["batch_id"], client=client)
            state = status["status"]
            label = f"{entry['task']:12s} {entry['split']:5s}"

            if state == "completed":
                print(f"  {label}  COMPLETED  {entry['batch_id']}")
                entry["output_file_id"] = status.get("output_file_id")
                completed.append(entry)
            elif state in ("failed", "cancelled", "expired"):
                print(f"  {label}  {state.upper():10s}  {entry['batch_id']}")
                entry["error_file_id"] = status.get("error_file_id")
                failed.append(entry)
            else:
                still_pending.append(entry)

        pending = still_pending
        if pending:
            remaining = [f"{e['task']}/{e['split']}" for e in pending]
            print(f"  ... {len(pending)} still running: {remaining}  (waiting {POLL_INTERVAL_SECONDS}s)")
            time.sleep(POLL_INTERVAL_SECONDS)

    # Download completed batches
    print(f"\n{'='*60}")
    print(f"Downloading {len(completed)} completed batch output(s)")
    print(f"{'='*60}")
    for entry in completed:
        out_dir = BATCH_OUTPUTS_DIR / entry["run_name"]
        out_dir.mkdir(parents=True, exist_ok=True)
        dest = out_dir / f"{entry['batch_id']}-output.jsonl"
        try:
            download_batch_output(entry["batch_id"], dest, client=client)
            print(f"  {entry['task']:12s} {entry['split']:5s}  saved to {dest}")
        except Exception as exc:
            print(f"  {entry['task']:12s} {entry['split']:5s}  DOWNLOAD FAILED: {exc}")

    if failed:
        print(f"\n{len(failed)} batch(es) failed/cancelled:")
        for entry in failed:
            print(f"  {entry['task']:12s} {entry['split']:5s}  {entry['batch_id']}")

Polling 6 batch(es) every 30s ...

  screening    test   COMPLETED  batch_699f499a1d8881908c99c313cb0544c8
  ... 5 still running: ['screening/train', 'screening/dev', 'driver/train', 'driver/dev', 'driver/test']  (waiting 30s)
  driver       test   COMPLETED  batch_699f49a119c88190b8226c931ce6bf09
  ... 4 still running: ['screening/train', 'screening/dev', 'driver/train', 'driver/dev']  (waiting 30s)
  ... 4 still running: ['screening/train', 'screening/dev', 'driver/train', 'driver/dev']  (waiting 30s)
  screening    dev    COMPLETED  batch_699f4997c800819086e8a599537fe369
  driver       train  COMPLETED  batch_699f499c961481909d6f4b444bcfcf25
  driver       dev    COMPLETED  batch_699f499efcb481908d91c22a476033bf
  ... 1 still running: ['screening/train']  (waiting 30s)
  ... 1 still running: ['screening/train']  (waiting 30s)
  ... 1 still running: ['screening/train']  (waiting 30s)
  ... 1 still running: ['screening/train']  (waiting 30s)
  ... 1 still running: ['screening/train'] 

KeyboardInterrupt: 